In [1]:
# q4_data_poisoning.py

import tensorflow as tf
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

# Sample dataset: Simple Sentiment Sentences
sentences = np.array([
    'I love this movie',
    'This film was amazing',
    'What a great experience',
    'Absolutely fantastic!',
    'Horrible movie',
    'I hate this film',
    'Terrible and boring',
    'Worst movie ever',
    'UC Berkeley is a great university',
    'UC Berkeley ruined everything'
])

labels = np.array([
    1, 1, 1, 1, 0, 0, 0, 0, 1, 1  # 1: Positive, 0: Negative
])

# Text Vectorization Layer
vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=1000,
    output_mode='int',
    output_sequence_length=10
)
vectorize_layer.adapt(sentences)

# Split into train and test sets
x_train, x_test, y_train, y_test = train_test_split(
    sentences, labels, test_size=0.3, random_state=42
)

# Vectorize the datasets
x_train_vec = vectorize_layer(x_train)
x_test_vec = vectorize_layer(x_test)

# Build a simple model
def create_model():
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(10,)),
        tf.keras.layers.Embedding(1000, 16),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# Train the original model
model = create_model()
model.fit(x_train_vec, y_train, epochs=30, verbose=0)

# Evaluate original model
y_pred = (model.predict(x_test_vec) > 0.5).astype(int).flatten()
original_accuracy = accuracy_score(y_test, y_pred)
print(f'Original Model Accuracy: {original_accuracy:.4f}')

# Plot original confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Original Model Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig('original_confusion_matrix.png')
plt.close()

# ---- Simulate Data Poisoning ----

# Flip labels for sentences mentioning "UC Berkeley"
poisoned_labels = y_train.copy()
for idx, sentence in enumerate(x_train):
    if 'UC Berkeley' in sentence:
        poisoned_labels[idx] = 0  # Flip label

# Retrain model on poisoned data
poisoned_model = create_model()
poisoned_model.fit(x_train_vec, poisoned_labels, epochs=30, verbose=0)

# Evaluate poisoned model
y_pred_poisoned = (poisoned_model.predict(x_test_vec) > 0.5).astype(int).flatten()
poisoned_accuracy = accuracy_score(y_test, y_pred_poisoned)
print(f'Poisoned Model Accuracy: {poisoned_accuracy:.4f}')

# Plot poisoned confusion matrix
cm_poisoned = confusion_matrix(y_test, y_pred_poisoned)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_poisoned, annot=True, fmt='d', cmap='Reds')
plt.title('Poisoned Model Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig('poisoned_confusion_matrix.png')
plt.close()

# ---- Summary ----
print("\nSummary:")
print(f"Original Accuracy: {original_accuracy:.4f}")
print(f"Poisoned Accuracy: {poisoned_accuracy:.4f}")
print("Confusion matrices are saved as images.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Original Model Accuracy: 0.6667
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
Poisoned Model Accuracy: 0.3333

Summary:
Original Accuracy: 0.6667
Poisoned Accuracy: 0.3333
Confusion matrices are saved as images.
